# DM887 GRPO for Control — Midway PoC (Claude Code candidate)

Candidate notebook generated by **Claude Code CLI** for the DM887 Reinforcement Learning term project.

This notebook is an **ObjectRL baseline experiment controller**. It does not reimplement
PPO, SAC, or TD3 — it orchestrates ObjectRL runs, captures logs, parses evaluation
returns, aggregates across seeds, and exports learning-curve figures suitable for the
midway report.

Three independent candidate notebooks (Claude Code, Copilot CLI, Codex) feed into the
final merged file `notebooks/DM887_Project_GRPO_Midway_PoC.ipynb`. **Do not overwrite
that merged notebook from this file.**

## 1. Assignment summary and midway scope

**Source of truth:** `DM887_Project.pdf` at the repository root.

### Final project goal
Design, analyse, and evaluate a **GRPO variant for control tasks**, starting from
vanilla Group Relative Policy Optimization, and compare it against PPO, SAC, and TD3
on the required control environments.

### Required environments
1. **Continuous Car Racing** — Gymnasium / Farama Box2D (instructor: use the
   continuous version; ignore the original reference to discrete-action envs).
2. **`cartpole-swingup-v0`** — DeepMind Control Suite.
3. **`acrobot-swingup-v0`** — DeepMind Control Suite.

### Evaluation protocol
- Online training with regular evaluation intervals.
- y-axis: **undiscounted evaluation episode return**.
- x-axis: **number of training steps before each evaluation**.
- Five seeds (`0, 1, 2, 3, 4`); seed values reported in the paper.
- Python + PyTorch implementation.

### Midway / interim scope (what this notebook supports)
1. **Related work** — covered in the report, not this notebook.
2. **MDP notation / formal setup** — covered in the report.
3. **PPO / SAC / TD3 baseline protocol and results** — *this notebook*.

ObjectRL is treated as the baseline implementation source for PPO, SAC, and TD3
(per instructor clarification). The final GRPO-for-control variant is **not**
implemented here — it belongs to the final-phase notebook.

## 2. Imports and paths

All paths are derived from the repository root. The notebook is robust to being
launched from either `notebooks/` or the repository root.

In [ ]:
from __future__ import annotations

import itertools
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# matplotlib only — no seaborn (per project policy)

In [ ]:
def find_repo_root(start: Path) -> Path:
    """Walk upward until we find DM887_Project.pdf or a .git directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "DM887_Project.pdf").exists():
            return candidate
        if (candidate / ".git").is_dir() and (candidate / "external").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OBJECTRL_DIR = REPO_ROOT / "external" / "objectrl"
GYMNASIUM_DIR = REPO_ROOT / "external" / "Gymnasium"

RESULTS_DIR = REPO_ROOT / "results"
RAW_RESULTS_DIR = RESULTS_DIR / "raw"
PROCESSED_RESULTS_DIR = RESULTS_DIR / "processed"
LOGS_DIR = RESULTS_DIR / "logs"
FIGURES_DIR = REPO_ROOT / "figures"
MIDWAY_FIGURES_DIR = FIGURES_DIR / "midway"

for d in (RAW_RESULTS_DIR, PROCESSED_RESULTS_DIR, LOGS_DIR, MIDWAY_FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT             :", REPO_ROOT)
print("OBJECTRL_DIR          :", OBJECTRL_DIR, "(exists:", OBJECTRL_DIR.exists(), ")")
print("GYMNASIUM_DIR         :", GYMNASIUM_DIR, "(exists:", GYMNASIUM_DIR.exists(), ")")
print("RAW_RESULTS_DIR       :", RAW_RESULTS_DIR)
print("PROCESSED_RESULTS_DIR :", PROCESSED_RESULTS_DIR)
print("LOGS_DIR              :", LOGS_DIR)
print("MIDWAY_FIGURES_DIR    :", MIDWAY_FIGURES_DIR)

## 3. Inspect ObjectRL

ObjectRL ships its CLI/config schema in `external/objectrl/objectrl/config/config.py`
and per-algorithm dataclasses in `external/objectrl/objectrl/config/model_configs/`.
These cells discover the available **model names**, **environment literals**, and the
exact CLI keys used by ObjectRL's `tyro`-based parser.

From a quick read of `config.py` the relevant CLI flags are:

| Field | CLI flag |
| --- | --- |
| Algorithm | `--model.name` |
| Environment | `--env.name` |
| Random seed | `--system.seed` *(NB: not `--seed`)* |
| Total steps | `--training.max_steps` |
| Eval episodes | `--training.eval_episodes` |
| Eval frequency | `--training.eval_frequency` |

**TODO — verify against the installed ObjectRL revision** before launching long runs;
the project's clarification only fixed the env list, not the CLI surface.

In [ ]:
def grep_objectrl(pattern: str, max_lines: int = 60) -> str:
    """Lightweight grep helper. Falls back to a Python scan if `grep` is missing."""
    if not OBJECTRL_DIR.exists():
        return f"[grep_objectrl] OBJECTRL_DIR not found: {OBJECTRL_DIR}"
    try:
        result = subprocess.run(
            ["grep", "-RIn", "--include=*.py", "--include=*.yaml", pattern, str(OBJECTRL_DIR)],
            capture_output=True,
            text=True,
            check=False,
        )
        lines = result.stdout.splitlines()
    except FileNotFoundError:
        rx = re.compile(pattern)
        lines = []
        for path in OBJECTRL_DIR.rglob("*.py"):
            try:
                for i, line in enumerate(path.read_text(errors="ignore").splitlines(), 1):
                    if rx.search(line):
                        lines.append(f"{path}:{i}:{line}")
            except OSError:
                continue
    return "\n".join(lines[:max_lines])


print("=== --env.name literals in ObjectRL config ===")
print(grep_objectrl(r"\"cheetah\"\|\"cartpole\"\|\"hopper\"\|\"dmc-\""))
print()
print("=== seed / system.seed references ===")
print(grep_objectrl(r"system\.seed\|\bseed:\b\|self\.seed\b"))

In [ ]:
MODEL_CONFIG_DIR = OBJECTRL_DIR / "objectrl" / "config" / "model_configs"
MODEL_YAML_DIR = OBJECTRL_DIR / "objectrl" / "config" / "model_yamls"

available_models: list[str] = []
if MODEL_CONFIG_DIR.exists():
    available_models = sorted(
        p.stem for p in MODEL_CONFIG_DIR.glob("*.py") if p.stem != "__init__"
    )

available_yamls: list[str] = []
if MODEL_YAML_DIR.exists():
    available_yamls = sorted(p.stem for p in MODEL_YAML_DIR.glob("*.yaml"))

print("ObjectRL model_configs/ :", available_models)
print("ObjectRL model_yamls/   :", available_yamls)

for required in ("ppo", "sac", "td3"):
    status = "OK" if required in available_models else "MISSING"
    print(f"  required: {required:<3}  ->  {status}")

## 4. Experiment matrix

Algorithms, environments, and seeds for the midway report. Three runtime modes
(`debug`, `midway`, `final`) trade off thoroughness against wall-clock time.

**Important:** ObjectRL's built-in `--env.name` literal list contains generic
MuJoCo / DMC names (`cheetah`, `cartpole`, `dmc-walker-run`, …) but **does not
include** `car-racing`, `cartpole-swingup-v0`, or `acrobot-swingup-v0` out of the
box. The mapping below therefore uses `"TODO_VERIFY_…"` placeholders. They must
be filled by either:

1. confirming that ObjectRL accepts a string outside its `Literal[...]` set
   (the `EnvConfig.name` field is typed `Literal[...] | str`, so it should), or
2. adding a small Gymnasium / DMC env wrapper inside ObjectRL, or
3. patching `EnvConfig` to add the three project envs.

Resolve this **before** kicking off long runs.

In [ ]:
ALGORITHMS: list[str] = ["ppo", "sac", "td3"]

PROJECT_ENVIRONMENTS: list[str] = [
    "car_racing_continuous",
    "cartpole_swingup",
    "acrobot_swingup",
]

# TODO: replace each placeholder with the verified --env.name string accepted
# by the ObjectRL revision in external/objectrl/. EnvConfig.name is typed
# Literal[...] | str, so non-listed names are syntactically allowed but the
# environment loader inside ObjectRL must actually know how to build them.
OBJECTRL_ENV_NAMES: dict[str, str] = {
    "car_racing_continuous": "TODO_VERIFY_CAR_RACING",
    "cartpole_swingup":      "TODO_VERIFY_CARTPOLE_SWINGUP",
    "acrobot_swingup":       "TODO_VERIFY_ACROBOT_SWINGUP",
}

SEEDS: list[int] = [0, 1, 2, 3, 4]

RUN_MODE: str = "debug"  # one of: debug | midway | final

RUN_CONFIG: dict[str, dict[str, Any]] = {
    "debug":  {"seeds": [0],         "max_steps":   1_000, "eval_episodes": 1, "eval_frequency":    500},
    "midway": {"seeds": [0,1,2,3,4], "max_steps":  20_000, "eval_episodes": 3, "eval_frequency":  2_000},
    "final":  {"seeds": [0,1,2,3,4], "max_steps": 500_000, "eval_episodes": 5, "eval_frequency": 20_000},
}

assert RUN_MODE in RUN_CONFIG, RUN_MODE
ACTIVE_CFG = RUN_CONFIG[RUN_MODE]
print(f"RUN_MODE = {RUN_MODE}")
print(json.dumps(ACTIVE_CFG, indent=2))

## 5. ObjectRL command builder

Builds the exact CLI invocation. The notebook calls ObjectRL via
`python -m objectrl.main` from inside `external/objectrl/`, which matches the
ObjectRL README's *“running from a cloned repo”* instructions.

The matrix is intentionally **exposed before any execution** so a dry-run pass
can print every command that would be issued.

In [ ]:
def build_objectrl_command(
    algorithm: str,
    project_env_name: str,
    seed: int,
    max_steps: int,
    eval_episodes: int,
    eval_frequency: int | None = None,
    extra_args: list[str] | None = None,
) -> tuple[list[str], str]:
    if algorithm not in ALGORITHMS:
        raise ValueError(f"Unknown algorithm: {algorithm!r}")
    if project_env_name not in OBJECTRL_ENV_NAMES:
        raise ValueError(f"Unknown project env: {project_env_name!r}")

    objectrl_env = OBJECTRL_ENV_NAMES[project_env_name]
    run_name = f"{algorithm}__{project_env_name}__seed{seed}__mode-{RUN_MODE}"

    cmd: list[str] = [
        sys.executable, "-m", "objectrl.main",
        "--model.name", algorithm,
        "--env.name", objectrl_env,
        "--system.seed", str(seed),
        "--training.max_steps", str(max_steps),
        "--training.eval_episodes", str(eval_episodes),
    ]
    if eval_frequency is not None:
        cmd += ["--training.eval_frequency", str(eval_frequency)]
    if extra_args:
        cmd += list(extra_args)

    return cmd, run_name


demo_cmd, demo_run = build_objectrl_command(
    "sac", "cartpole_swingup", seed=0,
    max_steps=ACTIVE_CFG["max_steps"],
    eval_episodes=ACTIVE_CFG["eval_episodes"],
    eval_frequency=ACTIVE_CFG["eval_frequency"],
)
print("run_name :", demo_run)
print("command  :", " ".join(demo_cmd))

## 6. Single-experiment runner

`run_experiment` performs one PPO/SAC/TD3 training run via subprocess, captures
stdout+stderr to a per-run log, and writes a `status.json` file. Failed runs are
recorded — never raised — so the orchestration loop covers the full matrix even
when one configuration breaks.

Set `dry_run=True` to print and persist the planned command without executing.

In [ ]:
def run_experiment(
    algorithm: str,
    project_env_name: str,
    seed: int,
    max_steps: int,
    eval_episodes: int,
    eval_frequency: int | None = None,
    dry_run: bool = False,
    extra_args: list[str] | None = None,
    timeout: float | None = None,
) -> dict[str, Any]:
    cmd, run_name = build_objectrl_command(
        algorithm=algorithm,
        project_env_name=project_env_name,
        seed=seed,
        max_steps=max_steps,
        eval_episodes=eval_episodes,
        eval_frequency=eval_frequency,
        extra_args=extra_args,
    )

    run_dir = RAW_RESULTS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    log_file = LOGS_DIR / f"{run_name}.log"
    status_file = run_dir / "status.json"

    status: dict[str, Any] = {
        "run_name": run_name,
        "algorithm": algorithm,
        "environment": project_env_name,
        "objectrl_env": OBJECTRL_ENV_NAMES[project_env_name],
        "seed": seed,
        "max_steps": max_steps,
        "eval_episodes": eval_episodes,
        "eval_frequency": eval_frequency,
        "run_mode": RUN_MODE,
        "command": " ".join(cmd),
        "cwd": str(OBJECTRL_DIR),
        "log_file": str(log_file),
        "status": "not_started",
        "return_code": None,
        "duration_seconds": None,
        "error": None,
    }

    if dry_run:
        status["status"] = "dry_run"
        status_file.write_text(json.dumps(status, indent=2))
        return status

    if not OBJECTRL_DIR.exists():
        status["status"] = "skipped_missing_objectrl"
        status["error"] = f"OBJECTRL_DIR not found: {OBJECTRL_DIR}"
        status_file.write_text(json.dumps(status, indent=2))
        return status

    start = time.time()
    try:
        with log_file.open("w") as fh:
            proc = subprocess.run(
                cmd,
                cwd=str(OBJECTRL_DIR),
                stdout=fh,
                stderr=subprocess.STDOUT,
                text=True,
                check=False,
                timeout=timeout,
            )
        status["return_code"] = proc.returncode
        status["status"] = "completed" if proc.returncode == 0 else "failed"
    except subprocess.TimeoutExpired as exc:
        status["status"] = "timeout"
        status["error"] = f"timeout after {exc.timeout}s"
    except Exception as exc:
        status["status"] = "error"
        status["error"] = repr(exc)
    finally:
        status["duration_seconds"] = time.time() - start
        status_file.write_text(json.dumps(status, indent=2))

    return status

## 7. Run order and dry-run preview

DMC envs first (cheaper than Car Racing), and within each env the
off-policy continuous-control baselines (SAC, TD3) before the on-policy PPO,
because PPO is more hyperparameter-sensitive and should not block the simpler
baselines from completing.

Set `DRY_RUN = True` first to confirm the full command set.

In [ ]:
RUN_ORDER: list[tuple[str, str]] = [
    ("sac", "cartpole_swingup"),
    ("td3", "cartpole_swingup"),
    ("ppo", "cartpole_swingup"),
    ("sac", "acrobot_swingup"),
    ("td3", "acrobot_swingup"),
    ("ppo", "acrobot_swingup"),
    ("sac", "car_racing_continuous"),
    ("td3", "car_racing_continuous"),
    ("ppo", "car_racing_continuous"),
]

DRY_RUN = True  # flip to False to actually launch ObjectRL

preview_rows: list[dict[str, Any]] = []
for algorithm, env_name in RUN_ORDER:
    for seed in ACTIVE_CFG["seeds"]:
        cmd, run_name = build_objectrl_command(
            algorithm=algorithm,
            project_env_name=env_name,
            seed=seed,
            max_steps=ACTIVE_CFG["max_steps"],
            eval_episodes=ACTIVE_CFG["eval_episodes"],
            eval_frequency=ACTIVE_CFG["eval_frequency"],
        )
        preview_rows.append({
            "run_name": run_name,
            "algorithm": algorithm,
            "environment": env_name,
            "seed": seed,
            "command": " ".join(cmd),
        })

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(PROCESSED_RESULTS_DIR / f"planned_runs_{RUN_MODE}.csv", index=False)
print(f"Planned runs ({len(preview_df)} configurations):")
preview_df.head(min(len(preview_df), 12))

## 8. Execute the matrix

Iterate `RUN_ORDER × seeds`. Status of every run is appended to a list and
exported to `results/processed/status_<mode>.csv`. Failures do not abort the
loop — they are recorded so the report can clearly distinguish completed runs
from interim/partial pilots.

In [ ]:
all_statuses: list[dict[str, Any]] = []

for algorithm, env_name in RUN_ORDER:
    for seed in ACTIVE_CFG["seeds"]:
        print(f"[{RUN_MODE}] {algorithm:<3} | {env_name:<22} | seed={seed} | dry_run={DRY_RUN}")
        status = run_experiment(
            algorithm=algorithm,
            project_env_name=env_name,
            seed=seed,
            max_steps=ACTIVE_CFG["max_steps"],
            eval_episodes=ACTIVE_CFG["eval_episodes"],
            eval_frequency=ACTIVE_CFG["eval_frequency"],
            dry_run=DRY_RUN,
        )
        all_statuses.append(status)
        print(f"   -> status={status['status']}  return_code={status['return_code']}  duration={status['duration_seconds']}")

status_df = pd.DataFrame(all_statuses)
status_csv = PROCESSED_RESULTS_DIR / f"status_{RUN_MODE}.csv"
status_json = PROCESSED_RESULTS_DIR / f"status_{RUN_MODE}.json"
status_df.to_csv(status_csv, index=False)
status_json.write_text(json.dumps(all_statuses, indent=2))
print(f"\nWrote {status_csv}")
print(f"Wrote {status_json}")
status_df[["run_name", "status", "return_code", "duration_seconds"]]

## 9. Parse evaluation returns from logs

ObjectRL's logging format depends on its loggers (TensorBoard, console, …).
The parser below tries several regular expressions; if your installed ObjectRL
writes structured CSV / TensorBoard event files instead of free-text console
lines, replace the regex pass with a direct file reader.

**TODO — open one log file in `results/logs/` after the first real run, then
tighten the regex against the actual line format.**

In [ ]:
# Patterns are deliberately broad. Order matters: the first match wins per line.
EVAL_LINE_PATTERNS: list[re.Pattern[str]] = [
    re.compile(r"step[^0-9]{0,8}(?P<step>\d+).*?(?:eval|evaluation).*?return[^0-9-]{0,8}(?P<ret>-?\d+(?:\.\d+)?)", re.IGNORECASE),
    re.compile(r"(?:eval|evaluation)[^0-9-]*step[^0-9]{0,8}(?P<step>\d+).*?return[^0-9-]{0,8}(?P<ret>-?\d+(?:\.\d+)?)", re.IGNORECASE),
    re.compile(r"\beval[^0-9-]{0,16}(?P<step>\d+)[^0-9-]{0,16}(?P<ret>-?\d+(?:\.\d+)?)", re.IGNORECASE),
]


def parse_log_file(log_file: Path) -> list[dict[str, float]]:
    """Return rows of {'step', 'evaluation_return'} extracted from one log."""
    if not log_file.exists():
        return []
    rows: list[dict[str, float]] = []
    for line in log_file.read_text(errors="ignore").splitlines():
        for pat in EVAL_LINE_PATTERNS:
            m = pat.search(line)
            if m:
                rows.append({
                    "step": int(m.group("step")),
                    "evaluation_return": float(m.group("ret")),
                })
                break
    return rows


def collect_evaluation_returns(statuses: list[dict[str, Any]]) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for st in statuses:
        log_path = Path(st["log_file"])
        for row in parse_log_file(log_path):
            records.append({
                "algorithm": st["algorithm"],
                "environment": st["environment"],
                "seed": st["seed"],
                "step": row["step"],
                "evaluation_return": row["evaluation_return"],
                "run_name": st["run_name"],
            })
    if not records:
        return pd.DataFrame(
            columns=["algorithm", "environment", "seed", "step", "evaluation_return", "run_name"]
        )
    return pd.DataFrame.from_records(records)


results_df = collect_evaluation_returns(all_statuses)
results_csv = PROCESSED_RESULTS_DIR / f"evaluation_returns_{RUN_MODE}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Parsed {len(results_df)} evaluation rows -> {results_csv}")
results_df.head()

## 10. Aggregate across seeds

Group by `(algorithm, environment, step)` and report mean / std / seed count.
Saved as `results/processed/evaluation_summary_<mode>.csv` for direct use in
the LaTeX report.

In [ ]:
if results_df.empty:
    summary_df = pd.DataFrame(
        columns=["algorithm", "environment", "step", "mean_return", "std_return", "n_seeds"]
    )
    print("No evaluation rows yet — summary is empty (expected in dry_run mode).")
else:
    summary_df = (
        results_df
        .groupby(["algorithm", "environment", "step"], as_index=False)
        .agg(
            mean_return=("evaluation_return", "mean"),
            std_return=("evaluation_return", "std"),
            n_seeds=("seed", "nunique"),
        )
        .sort_values(["environment", "algorithm", "step"])
    )

summary_csv = PROCESSED_RESULTS_DIR / f"evaluation_summary_{RUN_MODE}.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Wrote {summary_csv}")
summary_df.head()

## 11. Learning-curve plots

One subplot per environment, one line per algorithm.

- x-axis: training steps before evaluation
- y-axis: undiscounted evaluation episode return
- shaded band: ±1 std across seeds (only when ≥ 2 seeds reported)

Figures are exported as PDF + PNG into `figures/midway/` and copied with a
shared `midway_baselines` filename so the LaTeX report can `\includegraphics`
without renaming.

In [ ]:
ALGO_COLORS: dict[str, str] = {"ppo": "#1f77b4", "sac": "#2ca02c", "td3": "#d62728"}


def plot_learning_curves(summary: pd.DataFrame, output_stem: Path) -> Path | None:
    if summary.empty:
        print("plot_learning_curves: summary is empty — skipping figure export.")
        return None

    envs = [e for e in PROJECT_ENVIRONMENTS if e in set(summary["environment"])]
    envs += [e for e in summary["environment"].unique() if e not in envs]
    n = max(len(envs), 1)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, env_name in zip(axes, envs):
        env_df = summary[summary["environment"] == env_name]
        for algorithm in sorted(env_df["algorithm"].unique()):
            alg_df = env_df[env_df["algorithm"] == algorithm].sort_values("step")
            color = ALGO_COLORS.get(algorithm)
            ax.plot(
                alg_df["step"], alg_df["mean_return"],
                label=algorithm.upper(), color=color, linewidth=1.8,
            )
            if alg_df["std_return"].notna().any() and (alg_df["n_seeds"].max() or 0) >= 2:
                std = alg_df["std_return"].fillna(0.0)
                ax.fill_between(
                    alg_df["step"],
                    alg_df["mean_return"] - std,
                    alg_df["mean_return"] + std,
                    alpha=0.2, color=color, linewidth=0,
                )
        ax.set_title(env_name.replace("_", " "))
        ax.set_xlabel("Training steps")
        ax.set_ylabel("Undiscounted eval return")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize=9)

    fig.suptitle(f"PPO / SAC / TD3 — midway baselines ({RUN_MODE} mode)")
    fig.tight_layout()
    pdf_path = output_stem.with_suffix(".pdf")
    png_path = output_stem.with_suffix(".png")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved {pdf_path}")
    print(f"Saved {png_path}")
    return pdf_path


fig_stem = MIDWAY_FIGURES_DIR / f"midway_baselines_{RUN_MODE}"
plot_learning_curves(summary_df, fig_stem)

## 12. Report-facing notes

Generates a Markdown summary suitable for paste into the midway report's
experiments section. Lists completed runs, failed runs, the active run-mode
budget, and the list of artefacts produced.

In [ ]:
def build_report_notes(
    status_df: pd.DataFrame, summary_df: pd.DataFrame, mode: str
) -> str:
    completed = status_df[status_df["status"] == "completed"]
    failed    = status_df[status_df["status"].isin(["failed", "error", "timeout"])]
    dry       = status_df[status_df["status"] == "dry_run"]
    skipped   = status_df[status_df["status"].str.startswith("skipped")]

    lines: list[str] = []
    lines.append(f"# Midway baseline status — `{mode}` mode")
    lines.append("")
    lines.append("## Configuration")
    lines.append("")
    lines.append(f"- Algorithms: {ALGORITHMS}")
    lines.append(f"- Environments: {PROJECT_ENVIRONMENTS}")
    lines.append(f"- Seeds: {ACTIVE_CFG['seeds']}")
    lines.append(f"- max_steps: {ACTIVE_CFG['max_steps']}")
    lines.append(f"- eval_episodes: {ACTIVE_CFG['eval_episodes']}")
    lines.append(f"- eval_frequency: {ACTIVE_CFG['eval_frequency']}")
    lines.append("")
    lines.append("## Run accounting")
    lines.append("")
    lines.append(f"- planned runs : {len(status_df)}")
    lines.append(f"- completed    : {len(completed)}")
    lines.append(f"- failed/error : {len(failed)}")
    lines.append(f"- dry-run only : {len(dry)}")
    lines.append(f"- skipped      : {len(skipped)}")
    lines.append("")
    if not completed.empty:
        lines.append("### Completed runs")
        lines.append("")
        for _, row in completed.iterrows():
            lines.append(f"- `{row['run_name']}` ({row['duration_seconds']:.1f}s)")
        lines.append("")
    if not failed.empty:
        lines.append("### Failed / error / timeout")
        lines.append("")
        for _, row in failed.iterrows():
            lines.append(f"- `{row['run_name']}` — status={row['status']}, log={row['log_file']}")
        lines.append("")
    lines.append("## Evaluation summary")
    lines.append("")
    if summary_df.empty:
        lines.append("No evaluation rows parsed yet. The midway report should label these results as **interim / pilot**.")
    else:
        coverage = summary_df.groupby(["environment", "algorithm"])["n_seeds"].max().reset_index()
        for _, row in coverage.iterrows():
            lines.append(f"- {row['environment']} × {row['algorithm']}: up to {int(row['n_seeds'])} seeds")
    lines.append("")
    lines.append("## Generated artefacts")
    lines.append("")
    lines.append(f"- `{(PROCESSED_RESULTS_DIR / f'planned_runs_{mode}.csv').relative_to(REPO_ROOT)}`")
    lines.append(f"- `{(PROCESSED_RESULTS_DIR / f'status_{mode}.csv').relative_to(REPO_ROOT)}`")
    lines.append(f"- `{(PROCESSED_RESULTS_DIR / f'evaluation_returns_{mode}.csv').relative_to(REPO_ROOT)}`")
    lines.append(f"- `{(PROCESSED_RESULTS_DIR / f'evaluation_summary_{mode}.csv').relative_to(REPO_ROOT)}`")
    lines.append(f"- `{(MIDWAY_FIGURES_DIR / f'midway_baselines_{mode}.pdf').relative_to(REPO_ROOT)}`")
    lines.append(f"- `{(MIDWAY_FIGURES_DIR / f'midway_baselines_{mode}.png').relative_to(REPO_ROOT)}`")
    lines.append("")
    return "\n".join(lines)


notes = build_report_notes(status_df, summary_df, RUN_MODE)
notes_path = PROCESSED_RESULTS_DIR / f"report_notes_{RUN_MODE}.md"
notes_path.write_text(notes)
print(f"Wrote {notes_path}\n")
print(notes)

## 13. Open TODOs before running for the report

1. **Verify `--env.name` strings** for `car_racing_continuous`, `cartpole_swingup`,
   `acrobot_swingup`. ObjectRL's `EnvConfig.name` is `Literal[...] | str`; the
   environment loader inside `objectrl.experiments` must actually know how to
   build them. If not, add Gymnasium / DMC wrappers in ObjectRL — do **not**
   modify the cloned `external/objectrl/` source unless explicitly approved.
2. **Verify CLI keys** (`--system.seed`, `--training.eval_frequency`, …) against
   the installed ObjectRL revision; `--help_model <name>` and `--help` are the
   authoritative references.
3. **Tighten `EVAL_LINE_PATTERNS`** against an actual log produced by ObjectRL
   in this repo. If ObjectRL writes TensorBoard event files instead of console
   lines, replace the regex parser with a TB / CSV reader.
4. **Switch `DRY_RUN = False`** and `RUN_MODE = "midway"` once the env mapping
   and parser are verified. `final` mode (500k steps × 5 seeds × 9 configs) is
   for the final report, not the midway interim.
5. **Manual merge** — fold this candidate into
   `notebooks/DM887_Project_GRPO_Midway_PoC.ipynb` together with the Copilot CLI
   and Codex candidates per `plans/plan-poc.md` §8.